In [1]:
!pip install -U pypdf langchain_community chromadb langchain langchain_openai openai tiktoken rank_bm25 sentence_transformers cohere langchain_cohere

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.1/571.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:0

In [32]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
import os
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain.docstore.document import Document
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain_cohere import CohereRerank
import cohere
from langchain.document_loaders import PyPDFLoader
from google.colab import drive
from langchain.vectorstores import Chroma
import chromadb
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
import langchain

In [3]:
OPENAI_API_TOKEN=userdata.get('OPENAI_API_KEY')
COHERE_API_KEY = userdata.get('COHERE_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_TOKEN
os.environ["COHERE_API_KEY"] = COHERE_API_KEY

In [4]:
import textwrap
def wrap_text(text, width=90): #preserve_newlines
    # Split the input text into lines based on newline characters
    lines = text.split('\n')

    # Wrap each line individually
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]

    # Join the wrapped lines back together using newline characters
    wrapped_text = '\n'.join(wrapped_lines)

    return wrapped_text

In [5]:
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
loader_harrypotter  = PyPDFLoader("/content/harrypotter.pdf")
document_harrypotter = loader_harrypotter.load()
print(len(document_harrypotter))

7


In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [8]:
text_harrypotter = text_splitter.split_documents(document_harrypotter)
print(len(text_harrypotter))

101


In [9]:
embeddings = OpenAIEmbeddings()

In [10]:
os.getcwd()
CURRENT_DIR = os.path.dirname(os.path.abspath("."))
CURRENT_DIR
DB_DIR = os.path.join(CURRENT_DIR, "/content/db")
DB_DIR

'/content/db'

In [11]:
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [12]:
harrypotter_vectorstore = Chroma.from_documents(text_harrypotter,
                                       embeddings,
                                       client_settings=client_settings,
                                       collection_name="harrypotter",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/harrypotter")

In [13]:
retriever_harrypotter = harrypotter_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [14]:
llm_model = ChatOpenAI(model_name="gpt-4o-mini")

# Vanilla RAG Generation

In [16]:
qa = RetrievalQA.from_chain_type(
      llm=llm_model,
      chain_type="stuff",
      retriever = retriever_harrypotter,
      return_source_documents = True
)

In [17]:
query ="Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."
results = qa.invoke(query)

In [18]:
results['result']

"Harry Potter's best friends were Ron Weasley and Hermione Granger. Some traits Harry admired about them include:\n\n1. **Loyalty**: Both Ron and Hermione displayed unwavering loyalty to Harry throughout their adventures.\n2. **Bravery**: They showed courage in the face of danger, standing by Harry when confronting challenges.\n3. **Intelligence**: Hermione, in particular, was known for her cleverness and problem-solving skills, which often helped the trio navigate difficult situations.\n4. **Supportiveness**: They provided emotional support to Harry, especially during his struggles with his past and challenges against Voldemort.\n5. **Friendship**: Their willingness to form a deep, lasting bond distinguished them from other characters and highlighted the importance of friendship in the series."

# RAG Fusion

In [21]:
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.prompts import ChatMessagePromptTemplate, PromptTemplate, ChatPromptTemplate

In [22]:
prompt = ChatPromptTemplate(input_variables=['original_query'],
                            messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[],template='You are a helpful assistant that generates multiple search queries based on a single input query.')),
                            HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['original_query'], template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'))])


In [37]:
original_query = "Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."

In [38]:
generate_queries = (
    prompt | llm_model | StrOutputParser() | (lambda x: x.split("\n"))
)

In [39]:
generate_queries

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates multiple search queries based on a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7b50116d7a90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7b5010781010>, root_client=<openai.OpenAI object at 0x7b5011775910>, root_async_client=<openai.AsyncOpenAI object at 0x7b50116d6b50>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParse

In [27]:
from langchain.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        # Assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

In [29]:
ragfusion_chain = generate_queries | retriever_harrypotter.map() | reciprocal_rank_fusion

In [33]:
langchain.debug = True

In [35]:
ragfusion_chain.input_schema.model_json_schema()

{'properties': {'question': {'title': 'Question', 'type': 'string'}},
 'required': ['question'],
 'title': 'PromptInput',
 'type': 'object'}

In [40]:
ragfusion_chain.invoke({"question": original_query})

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "System: You are a helpful assistant that generates multiple search queries based on a single input query.\nHuman: Generate multiple search queries related to: Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them. \n OUTPUT (4 queries):"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generation

[(Document(metadata={'creationdate': '2020-07-14T01:46:48+08:00', 'creator': 'Adobe InDesign CS6 (Windows)', 'moddate': '2020-07-14T01:48:33+08:00', 'page': 3, 'page_label': '4', 'producer': 'Adobe PDF Library 10.0.1', 'source': '/content/harrypotter.pdf', 'total_pages': 7}, page_content='factors leading to the success of Harry Potter serious. \nHarry’s friendship with Ron and Hermione was \nalso significant in the way that it further distinguished \nHarry from V oldemort. Although V oldemort was far \nmore powerful than Harry, he preferred to be isolated \nand independent from those around him. Even Professor \nQuirrell, who drunk unicorn blood for him, was nothing \nmore than a servant to V oldemort. V oldemort lacked the \nability to form lasting friendships, so he was always alone'),
  0.06639344262295081),
 (Document(metadata={'creationdate': '2020-07-14T01:46:48+08:00', 'creator': 'Adobe InDesign CS6 (Windows)', 'moddate': '2020-07-14T01:48:33+08:00', 'page': 6, 'page_label': '7'

In [41]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

full_rag_fusion_chain = (
    {
        "context": ragfusion_chain,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm_model
    | StrOutputParser()
)

In [43]:
full_rag_fusion_chain.input_schema.model_json_schema()

{'properties': {'question': {'title': 'Question', 'type': 'string'},
  'root': {'title': 'Root'}},
 'required': ['question', 'root'],
 'title': 'RunnableParallel<context,question>Input',
 'type': 'object'}

In [44]:
full_rag_fusion_chain.invoke({"question": "Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."})

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "question": "Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."
}
[chain/end] [chain:RunnableSequence > chain:Runn

"Harry Potter's best friends were Ron Weasley and Hermione Granger. Harry admired Ron's bravery, particularly when Ron sacrificed himself during a life-sized game of wizard's chess, and he appreciated Hermione's cleverness and loyalty. Additionally, their friendship distinguished Harry from Voldemort, who was isolated and lacked the ability to form lasting relationships."